In [661]:
import heapq
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import collections
import itertools
import tqdm
from copy import deepcopy

In [662]:
np.random.seed(0)

In [663]:
def balance_priors(priors, random=True):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [664]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [665]:
def manipulation_thresholds(thresholds, priors, c):
    if np.sum(priors) == 0:
        return thresholds.copy()
    manip_thresholds = np.maximum(0, thresholds - (bayesian_update(priors) / c))
    return manip_thresholds

In [666]:
def merge_classifiers(thresholds, priors, manip_thresholds):
    if len(thresholds) <= 1:
        return thresholds, priors, manip_thresholds
    
    merged_thresholds = [thresholds[-1]]
    merged_priors = [priors[-1]]
    merged_manip_thresholds = [manip_thresholds[-1]]

    for i in range(len(thresholds)-2,-1,-1):
        if manip_thresholds[i] < merged_manip_thresholds[-1]:
            merged_thresholds.append(thresholds[i])
            merged_priors.append(priors[i])
            merged_manip_thresholds.append(manip_thresholds[i])
        else:
            merged_priors[-1] += priors[i]
    
    return np.array(merged_thresholds)[::-1], np.array(merged_priors)[::-1], np.array(merged_manip_thresholds)[::-1]

In [667]:
def accuracy_loss(thresholds, priors, manip_thresholds, threshold_true):
    losses = []
    for i in range(len(thresholds)):
        loss = np.abs(manip_thresholds[i] - threshold_true)
        losses.append(loss)
    losses = np.array(losses)
    posteriors = bayesian_update(priors)
    return np.dot(losses, posteriors)

In [668]:
def evaluate_partition(partition, thresholds, priors, threshold_true, c, return_all=False):
    thresholds_p = thresholds[partition]
    priors_p = priors[partition]
    manip_thresholds_p = manipulation_thresholds(thresholds_p, priors_p, c)
    thresholds_p, priors_p, manip_thresholds_p = merge_classifiers(thresholds_p, priors_p, manip_thresholds_p)
    acc_loss_p = accuracy_loss(thresholds_p, priors_p, manip_thresholds_p, threshold_true)
    if return_all:
        return acc_loss_p, thresholds_p, priors_p, manip_thresholds_p
    return acc_loss_p

def evaluate_system(partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [669]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [670]:
def display_queue(Q, P):
    res = "[  "
    for (a_id, b_id) in Q:
        res += f"({P[a_id]}, {P[b_id]})  "
    res += "]"
    print(res)


def find_partitions_greedy_lex(thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1

    Q = collections.deque(itertools.combinations(P.keys(), 2))
    while Q:
        if display:
            display_queue(Q, P)
        a_id, b_id = Q.popleft()
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        acc_loss_ab = evaluate_partition(ab, thresholds, priors, threshold_true, c)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs > rhs:
            del P[a_id]
            del P[b_id]

            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    Q.append((new_id, p_id))

            # Q = collections.deque(sorted(Q, key=lambda x: P[x[0]]))
            
    return list(P.values())

In [671]:
def display_priority_queue(pq, P):
    res = "[  "
    for acc_loss, (a_id, b_id) in pq:
        res += f"({acc_loss:.4f}, ({P[a_id]}, {P[b_id]}))  "
    res += "]"
    print(res)


def find_partitions_greedy_best(thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = block
        next_id += 1
    
    Q = collections.deque(itertools.combinations(P.keys(), 2))
    pq = []
    for a_id, b_id in Q:
        acc_loss_ab = evaluate_partition(sorted(P[a_id]+P[b_id]), thresholds, priors, threshold_true, c)
        heapq.heappush(pq, (acc_loss_ab, (a_id, b_id)))

    while pq:
        if display:
            display_priority_queue(pq, P)
        acc_loss_ab, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs > rhs:
            del P[a_id]
            del P[b_id]

            pq2 = []
            for acc_loss, (x_id, y_id) in pq:
                if x_id not in {a_id, b_id} and y_id not in {a_id, b_id}:
                    acc_loss = evaluate_partition(sorted(P[x_id]+P[y_id]), thresholds, priors, threshold_true, c)
                    heapq.heappush(pq2, (acc_loss, (x_id,y_id)))
            pq = deepcopy(pq2)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    acc_loss = evaluate_partition(sorted(P[p_id]+P[new_id]), thresholds, priors, threshold_true, c)
                    heapq.heappush(pq, (acc_loss, (new_id, p_id)))
    return list(P.values())

In [672]:
def find_partitions_optimal(thresholds, priors, threshold_true, c):
    indices = [i for i in range(len(thresholds))]

    parts = set_partitions(indices)
    partitions_set = []
    for part in parts:
        partitions_set.append(part)

    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.
        for partition in partitions:
            acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
            acc_loss += acc_loss_p * np.sum(priors[partition])

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = deepcopy(partitions)

    return best_partition

In [707]:
threshold_true = 0.5

threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
thresholds = np.array([0.2, 0.4, 0.6, 0.8])
priors = np.array([1/4, 1/4, 1/4, 1/4])

# threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
# thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)
# priors = np.zeros_like(thresholds)
# balance_priors(priors, random=True)
# priors = np.array([0.11393634, 0.11459784, 0.03594993, 0.25, 0.02203078, 0.05390004, 0.06215049, 0.09743458, 0.25])

In [708]:
C = np.arange(0.1, 20, 0.1)
# C = [50000.]

regularizer = []
accuracy_losses_greedy_lex = []
accuracy_losses_optimal = []

for c in tqdm.tqdm(C):
    partition_greedy_lex = find_partitions_greedy_lex(thresholds, priors, threshold_true, c, len(C)==1)
    partition_optimal = find_partitions_optimal(thresholds, priors, threshold_true, c)

    acc_loss_greedy_lex = evaluate_system(partition_greedy_lex, thresholds, priors, threshold_true, c)
    acc_loss_optimal = evaluate_system(partition_optimal, thresholds, priors, threshold_true, c)

    regularizer.append(c)
    accuracy_losses_greedy_lex.append(acc_loss_greedy_lex.item())
    accuracy_losses_optimal.append(acc_loss_optimal.item())

100%|██████████| 199/199 [00:00<00:00, 1200.18it/s]


In [709]:
ratio_lex = []
for i in range(len(C)):
    ratio_lex.append((1 - accuracy_losses_optimal[i]) / (1 - accuracy_losses_greedy_lex[i]))

In [710]:
results = {"c": regularizer, "greedy_lex": accuracy_losses_greedy_lex, "optimal": accuracy_losses_optimal}
px.line(results, x="c", y=["greedy_lex", "optimal"], markers=len(C)==1).update_layout(yaxis=dict(title="Accuracy Loss"))

In [711]:
results_ratio = {"c": C, "ratio_lex": ratio_lex}
px.line(results_ratio, x="c", y="ratio_lex", markers=len(C)==1)

In [702]:
print("Greedy Lexicographic")
print("------")
print(f"Partition: {sorted(partition_greedy_lex)}")
print(f"Acc Loss : {acc_loss_greedy_lex:.4f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_optimal)}")
print(f"Acc Loss : {acc_loss_optimal:.4f}")

Greedy Lexicographic
------
Partition: [[0, 1, 2, 3], [4, 8], [5], [6], [7]]
Acc Loss : 0.2592

Optimal
-------
Partition: [[0, 1, 2, 3, 4], [5], [6], [7], [8]]
Acc Loss : 0.2592


In [706]:
a, b = [0,1,2,3,4,5,6,8], [7]

acc_loss_a, thresholds_a, priors_a, manip_thresholds_a = evaluate_partition(a, thresholds, priors, threshold_true, c, True)
acc_loss_b, thresholds_b, priors_b, manip_thresholds_b = evaluate_partition(b, thresholds, priors, threshold_true, c, True)

lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
ab = sorted(a + b)

acc_loss_ab, thresholds_ab, priors_ab, manip_thresholds_ab = evaluate_partition(ab, thresholds, priors, threshold_true, c, True)
rhs = acc_loss_ab * np.sum(priors[ab])

print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.4f}")
print(f"   accuracy loss b: {acc_loss_b:.4f}")
print(f"  accuracy loss ab: {acc_loss_ab:.4f}")
print(f"               LHS: {lhs:.4f}")
print(f"               RHS: {rhs:.4f}")
print(f"            merge?: {lhs>rhs}")
print()
print(f"      thresholds a: {thresholds_a.round(4)}")
print(f"          priors a: {priors_a.round(4)}")
print(f" manip threshold a: {manip_thresholds_a.round(4)}")
print(f"      thresholds b: {thresholds_b.round(4)}")
print(f"          priors b: {priors_b.round(4)}")
print(f" manip threshold b: {manip_thresholds_b.round(4)}")
print(f"     thresholds ab: {thresholds_ab.round(4)}")
print(f"         priors ab: {priors_ab.round(4)}")
print(f"manip threshold ab: {manip_thresholds_ab.round(4)}")

                 c: 50000.0
                 a: [0, 1, 2, 3, 4, 5, 6, 8]
                 b: [7]
   accuracy loss a: 0.2548
   accuracy loss b: 0.3000
  accuracy loss ab: 0.2592
               LHS: 0.2592
               RHS: 0.2592
            merge?: False

      thresholds a: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.9]
          priors a: [0.1139 0.1146 0.0359 0.25   0.022  0.0539 0.0622 0.25  ]
 manip threshold a: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.9]
      thresholds b: [0.8]
          priors b: [0.0974]
 manip threshold b: [0.8]
     thresholds ab: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
         priors ab: [0.1139 0.1146 0.0359 0.25   0.022  0.0539 0.0622 0.0974 0.25  ]
manip threshold ab: [0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
